# Clustering Enhanced Binary Models for Valence and Arousal Prediction

This notebook develops a two layer global modeling approach for predicting self-reported valence and arousal from video level sensor derived features.

The modeling task is formulated as binary classification. Numerical valence and arousal ratings are first converted into low, medium and high categories. For the binary task, only low and high samples are retained, while medium samples are excluded. This creates a clearer classification problem and focuses the analysis on stronger affective differences.

The model architecture contains two stages. First, an unsupervised clustering layer is used to identify latent response patterns from the sensor features. Then, supervised machine learning classifiers use the sensor features together with the cluster derived representation to predict low versus high valence or arousal.

All models are evaluated using Leave One Participant Out cross validation. This ensures that all samples from the same participant are held out together and prevents participant leakage. Scene identity is not used as an input feature, because the goal is to evaluate whether sensor-derived features and latent response patterns can predict affective ratings.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

from sklearn.cluster import KMeans

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

## 1. Loading the Clean Feature Dataset

The cleaned video-level feature dataset is loaded from the project output directory. Each row corresponds to one participant watching one empathy scene. The dataset contains self-reported valence and arousal ratings together with sensor-derived minimum and maximum features extracted from the corresponding scene window.

In [2]:
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = None
for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (candidate / "outputs").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root. Make sure the notebook is inside the project folder."
    )

OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_PATH = OUTPUT_DIR / "video_level_valence_arousal_minmax.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)

display(df.head())

Project root: /Users/victoriaprojkova/EmteqPRO-VR
Dataset path: /Users/victoriaprojkova/EmteqPRO-VR/outputs/video_level_valence_arousal_minmax.csv
Dataset shape: (384, 57)


,participant_id,participant_folder,recording_suffix,video_set,scene_name,scene_index,target_valence,target_arousal,target_discomfort,window_start,window_end,window_duration_seconds,segment_duration_seconds,trim_seconds_used,timing_source,feature_extraction_strategy,breathingrate_breathingrate_min,breathingrate_breathingrate_max,breathingrate_imu_motionintensity_min,breathingrate_imu_motionintensity_max,qc_breathingrate_ppg_quality_min,qc_breathingrate_ppg_quality_mean,emgactivation_emg_amplitude_zygo_weighted_min,emgactivation_emg_amplitude_zygo_weighted_max,emgactivation_emg_amplitude_orbi_weighted_min,emgactivation_emg_amplitude_orbi_weighted_max,emgactivation_emg_amplitude_front_weighted_min,emgactivation_emg_amplitude_front_weighted_max,emgactivation_emg_amplitude_corr_weighted_min,emgactivation_emg_amplitude_corr_weighted_max,expression_expression_intensity_min,expression_expression_intensity_max,expression_neutral_intensity_min,expression_neutral_intensity_max,facialactivation_facialactivation_min,facialactivation_facialactivation_max,hrv_hrv_mean_hr_min,hrv_hrv_mean_hr_max,hrv_hrv_rr_min,hrv_hrv_rr_max,hrv_hrv_sdnn_min,hrv_hrv_sdnn_max,hrv_hrv_sdsd_min,hrv_hrv_sdsd_max,hrv_hrv_rmssd_min,hrv_hrv_rmssd_max,hrv_imu_motionintensity_min,hrv_imu_motionintensity_max,qc_hrv_ppg_quality_min,qc_hrv_ppg_quality_mean,expression_smile_intensity_min,expression_smile_intensity_max,expression_eyebrow_raise_intensity_min,expression_eyebrow_raise_intensity_max,expression_frown_intensity_min,expression_frown_intensity_max,n_core_available_features
0,3,participant3,0,3,empathy_scene_1,1,1,3,1,1.704461e+09,1.704461e+09,120.0,200.0,40.0,details_start_end,split6_trimmed_fallback,10.446816,15.640528,0.0,0.846154,1.000000,1.000000,0.471619,74.557578,2.989772,193.801556,0.542198,28.991436,0.585319,16.507440,0.0,21.9,0.0,0.0,0.030521,0.425829,65.308989,73.409462,817.333333,918.709677,55.167892,139.210472,64.498062,177.132665,64.498062,177.180991,0.0,0.733333,1.0,1.0,5.5,21.9,0.0,0.0,7.4,8.3,28
1,3,participant3,0,3,empathy_scene_2,2,3,3,2,1.704461e+09,1.704461e+09,120.0,200.0,40.0,details_start_end,split6_trimmed_fallback,10.373120,15.919705,0.0,0.230769,1.000000,1.000000,0.758779,93.319921,2.326091,367.574970,-0.003053,59.168081,0.374244,12.118754,0.0,30.0,0.0,0.0,0.023948,0.722539,61.354020,75.227964,797.575758,977.931034,62.136443,122.216269,68.605959,132.837023,69.282032,132.842233,0.0,0.200000,1.0,1.0,4.4,30.0,13.1,21.3,0.0,0.0,28
2,3,participant3,0,3,empathy_scene_3,3,0,0,0,1.704461e+09,1.704461e+09,120.0,200.0,40.0,details_start_end,split6_trimmed_fallback,9.163744,15.569518,0.0,0.230769,1.000000,1.000000,4.026037,81.549640,12.058416,384.159678,-0.109454,63.268512,0.564494,23.403621,0.0,29.9,0.0,0.0,0.083460,0.664852,61.179898,72.727273,825.000000,980.714286,52.136573,119.990855,67.656543,157.585330,67.670602,158.065225,0.0,0.200000,1.0,1.0,4.5,29.9,0.0,0.0,0.0,0.0,28
3,3,participant3,0,3,empathy_scene_4,4,2,1,1,1.704461e+09,1.704461e+09,120.0,200.0,40.0,details_start_end,split6_trimmed_fallback,10.739601,14.466929,0.0,0.461538,0.978226,0.999453,2.112242,280.625786,7.264172,514.008341,-0.022390,24.817357,0.500666,14.036135,0.0,108.1,0.0,0.0,0.055702,1.687703,61.009818,69.915254,858.181818,983.448276,52.746775,123.979387,73.142475,158.307160,73.155479,158.401102,0.0,0.266667,1.0,1.0,5.0,108.1,0.0,0.0,0.0,0.0,28
4,4,participant4,0,1,empathy_scene_1,1,4,2,2,1.704737e+09,1.704737e+09,130.0,210.0,40.0,details_start_end,split6_trimmed_fallback,10.039688,14.118108,0.0,0.000000,1.000000,1.000000,2.737313,93.411802,0.376880,315.791773,0.190412,47.383158,0.851178,77.210891,0.0,0.0,0.0,0.0,0.027433,0.124473,89.743590,99.858357,600.851064,668.571429,23.385684,61.159173,28.791068,39.553271,28.919952,39.562828,0.0,0.000000,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,28


In [3]:
print("Dataset shape:", df.shape)

print("\nNumber of participants:")
print(df["participant_id"].nunique())

print("\nRows by scene:")
display(df["scene_name"].value_counts().sort_index().to_frame("n_rows"))

print("\nRows by participant:")
rows_by_participant = (
    df.groupby(["participant_id", "participant_folder"])
    .size()
    .to_frame("n_rows")
    .reset_index()
)

display(rows_by_participant["n_rows"].describe())

print("\nMissing targets:")
target_cols = ["target_valence", "target_arousal"]

if "target_discomfort" in df.columns:
    target_cols.append("target_discomfort")

display(df[target_cols].isna().sum().to_frame("n_missing"))

Dataset shape: (384, 57)

Number of participants:
96

Rows by scene:


,n_rows
scene_name,
empathy_scene_1,96
empathy_scene_2,96
empathy_scene_3,96
empathy_scene_4,96



Rows by participant:


count    96.0
mean      4.0
std       0.0
min       4.0
25%       4.0
50%       4.0
75%       4.0
max       4.0
Name: n_rows, dtype: float64


Missing targets:


,n_missing
target_valence,0
target_arousal,0
target_discomfort,0


## 2. Binary Target Definition

The numerical valence and arousal ratings are converted into low, medium and high categories. Ratings from 0 to 1 are treated as low, rating 2 is treated as medium, and ratings greater than or equal to 3 are treated as high.

For binary classification, only low and high samples are retained. Medium samples are excluded because they represent an intermediate response level and may be less clearly separable from the sensor-derived features.

In [4]:
def bin_low_medium_high(x):
    if pd.isna(x):
        return np.nan
    if x <= 1:
        return "low"
    if x == 2:
        return "medium"
    if x >= 3:
        return "high"
    return "unknown"


df["valence_category"] = df["target_valence"].apply(bin_low_medium_high)
df["arousal_category"] = df["target_arousal"].apply(bin_low_medium_high)

category_order = ["low", "medium", "high"]

print("Valence category counts:")
display(
    df["valence_category"]
    .value_counts()
    .reindex(category_order)
    .to_frame("count")
)

print("Arousal category counts:")
display(
    df["arousal_category"]
    .value_counts()
    .reindex(category_order)
    .to_frame("count")
)

Valence category counts:


,count
valence_category,
low,88
medium,82
high,214


Arousal category counts:


,count
arousal_category,
low,121
medium,131
high,132


In [5]:
metadata_cols = {
    "participant_id",
    "participant_folder",
    "recording_suffix",
    "video_set",
    "scene_name",
    "scene_index",
    "target_valence",
    "target_arousal",
    "target_discomfort",
    "window_start",
    "window_end",
    "window_duration_seconds",
    "segment_duration_seconds",
    "trim_seconds_used",
    "timing_source",
    "feature_extraction_strategy",
    "valence_category",
    "arousal_category",
    "discomfort_category",
    "n_core_available_features",
}

feature_cols = [
    col for col in df.columns
    if col not in metadata_cols
    and not col.startswith("qc_")
]

print("Number of original sensor feature columns:", len(feature_cols))

display(pd.DataFrame({"feature": feature_cols}))

Number of original sensor feature columns: 36


,feature
0,breathingrate_breathingrate_min
1,breathingrate_breathingrate_max
2,breathingrate_imu_motionintensity_min
3,breathingrate_imu_motionintensity_max
4,emgactivation_emg_amplitude_zygo_weighted_min
5,emgactivation_emg_amplitude_zygo_weighted_max
6,emgactivation_emg_amplitude_orbi_weighted_min
7,emgactivation_emg_amplitude_orbi_weighted_max
8,emgactivation_emg_amplitude_front_weighted_min
9,emgactivation_emg_amplitude_front_weighted_max


## 3. Enhanced Sensor Feature Representation

The original dataset contains minimum and maximum values for each sensor-derived feature. To provide the models with a richer representation, mean and range features are created from each matching minimum-maximum pair.

The mean feature represents the average level of the signal within the scene window, while the range feature captures the spread between the minimum and maximum values. This step uses only input features and does not use target information.

In [6]:
X_base = df[feature_cols].copy()
X_enhanced = X_base.copy()

for col in X_base.columns:
    if col.endswith("_min"):
        base = col.replace("_min", "")
        max_col = base + "_max"

        if max_col in X_base.columns:
            X_enhanced[f"{base}_mean"] = (
                X_base[col] + X_base[max_col]
            ) / 2

            X_enhanced[f"{base}_range"] = (
                X_base[max_col] - X_base[col]
            )

print("Original features:", X_base.shape[1])
print("Enhanced features:", X_enhanced.shape[1])
print("Added features:", X_enhanced.shape[1] - X_base.shape[1])

display(pd.DataFrame({"feature": X_enhanced.columns}))

Original features: 36
Enhanced features: 72
Added features: 36


,feature
0,breathingrate_breathingrate_min
1,breathingrate_breathingrate_max
2,breathingrate_imu_motionintensity_min
3,breathingrate_imu_motionintensity_max
4,emgactivation_emg_amplitude_zygo_weighted_min
5,emgactivation_emg_amplitude_zygo_weighted_max
6,emgactivation_emg_amplitude_orbi_weighted_min
7,emgactivation_emg_amplitude_orbi_weighted_max
8,emgactivation_emg_amplitude_front_weighted_min
9,emgactivation_emg_amplitude_front_weighted_max


## 4. Preparing Binary Valence and Arousal Datasets

Separate binary datasets are created for valence and arousal. For each target, medium samples are removed and the remaining low and high samples are encoded as binary labels.

The participant identifier is preserved and used as the grouping variable for Leave-One-Participant-Out cross-validation.

In [7]:
binary_label_mapping = {
    "low": 0,
    "high": 1,
}

inverse_binary_label_mapping = {
    0: "low",
    1: "high",
}

binary_label_order_numeric = [0, 1]
binary_label_order_names = ["low", "high"]


def prepare_binary_dataset(df, X, target_category_col):
    binary_mask = df[target_category_col].isin(["low", "high"])

    X_binary = X.loc[binary_mask].reset_index(drop=True)

    y_binary = (
        df.loc[binary_mask, target_category_col]
        .map(binary_label_mapping)
        .reset_index(drop=True)
    )

    groups_binary = (
        df.loc[binary_mask, "participant_id"]
        .reset_index(drop=True)
    )

    metadata_binary = (
        df.loc[
            binary_mask,
            [
                "participant_id",
                "participant_folder",
                "scene_name",
                "target_valence",
                "target_arousal",
                "valence_category",
                "arousal_category",
            ],
        ]
        .reset_index(drop=True)
    )

    return X_binary, y_binary, groups_binary, metadata_binary

In [8]:
X_valence_binary, y_valence_binary, groups_valence_binary, valence_binary_metadata = prepare_binary_dataset(
    df=df,
    X=X_enhanced,
    target_category_col="valence_category",
)

X_arousal_binary, y_arousal_binary, groups_arousal_binary, arousal_binary_metadata = prepare_binary_dataset(
    df=df,
    X=X_enhanced,
    target_category_col="arousal_category",
)

print("Valence binary dataset:")
print("X shape:", X_valence_binary.shape)
print("Participants:", groups_valence_binary.nunique())
display(
    y_valence_binary
    .value_counts()
    .sort_index()
    .rename(index=inverse_binary_label_mapping)
    .to_frame("count")
)

print("\nArousal binary dataset:")
print("X shape:", X_arousal_binary.shape)
print("Participants:", groups_arousal_binary.nunique())
display(
    y_arousal_binary
    .value_counts()
    .sort_index()
    .rename(index=inverse_binary_label_mapping)
    .to_frame("count")
)

Valence binary dataset:
X shape: (302, 72)
Participants: 96


,count
valence_category,
low,88
high,214



Arousal binary dataset:
X shape: (253, 72)
Participants: 95


,count
arousal_category,
low,121
high,132


In [9]:
logo = LeaveOneGroupOut()

n_valence_folds = logo.get_n_splits(
    X_valence_binary,
    y_valence_binary,
    groups_valence_binary,
)

n_arousal_folds = logo.get_n_splits(
    X_arousal_binary,
    y_arousal_binary,
    groups_arousal_binary,
)

print("Valence LOPO folds:", n_valence_folds)
print("Arousal LOPO folds:", n_arousal_folds)

Valence LOPO folds: 96
Arousal LOPO folds: 95


## 5. Clustering-Based Feature Layer

This section defines the first layer of the two-layer architecture. The clustering layer is unsupervised and is used to learn latent response patterns from the sensor-derived features.

KMeans clustering is fitted only on the training data within each Leave-One-Participant-Out fold. For each sample, the distances to the learned cluster centroids are computed and added to the original feature representation. These cluster-distance features provide the supervised classifier with information about how close each sample is to different latent response patterns.

The clustering step is included inside the machine learning pipeline to avoid data leakage.

In [10]:
from sklearn.utils.validation import check_is_fitted

In [11]:
class KMeansDistanceFeatureAugmenter(BaseEstimator, TransformerMixin):
    """
    Adds distances to KMeans cluster centroids as additional features.

    The transformer is designed to be used inside a cross-validation
    pipeline. KMeans is fitted only on the training fold, and the test
    fold is transformed using the centroids learned from the training fold.
    """

    def __init__(self, n_clusters=3, random_state=42, n_init=20):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.n_init = n_init

    def fit(self, X, y=None):
        X = np.asarray(X)

        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=self.n_init,
        )

        self.kmeans_.fit(X)

        return self

    def transform(self, X):
        check_is_fitted(self, "kmeans_")

        X = np.asarray(X)

        cluster_distances = self.kmeans_.transform(X)

        X_augmented = np.hstack(
            [
                X,
                cluster_distances,
            ]
        )

        return X_augmented

In [12]:
quick_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
        ("scaler", StandardScaler()),
    ]
)

X_valence_preprocessed = quick_preprocessor.fit_transform(X_valence_binary)

cluster_augmenter = KMeansDistanceFeatureAugmenter(
    n_clusters=3,
    random_state=42,
)

X_valence_cluster_augmented = cluster_augmenter.fit_transform(
    X_valence_preprocessed
)

print("Original valence binary shape:", X_valence_binary.shape)
print("Preprocessed valence shape:", X_valence_preprocessed.shape)
print("Cluster-augmented valence shape:", X_valence_cluster_augmented.shape)

Original valence binary shape: (302, 72)
Preprocessed valence shape: (302, 65)
Cluster-augmented valence shape: (302, 68)


In [13]:
def make_clustering_model_pipeline(
    classifier,
    n_clusters=3,
    sampler=None,
    use_feature_selection=False,
    k_best=30,
):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
        ("scaler", StandardScaler()),

        # Layer 1: unsupervised clustering representation
        ("cluster_features", KMeansDistanceFeatureAugmenter(
            n_clusters=n_clusters,
            random_state=42,
            n_init=20,
        )),

        # Re-scale after adding cluster-distance features
        ("post_cluster_scaler", StandardScaler()),
    ]

    if use_feature_selection:
        steps.append(
            ("feature_selection", SelectKBest(
                score_func=f_classif,
                k=k_best,
            ))
        )

    if sampler is not None:
        steps.append(("sampler", sampler))
        steps.append(("classifier", classifier))
        return ImbPipeline(steps=steps)

    steps.append(("classifier", classifier))
    return Pipeline(steps=steps)

In [14]:
#the best performative models from the previous notebook analysis are chosen

def build_clustering_ml_models(
    n_clusters=3,
    sampler=None,
    use_feature_selection=False,
    k_best=30,
):
    models = {
        "Dummy most frequent": make_clustering_model_pipeline(
            DummyClassifier(strategy="most_frequent"),
            n_clusters=n_clusters,
            sampler=None,
            use_feature_selection=False,
        ),

        "Logistic Regression": make_clustering_model_pipeline(
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42,
            ),
            n_clusters=n_clusters,
            sampler=sampler,
            use_feature_selection=use_feature_selection,
            k_best=k_best,
        ),

        "RBF SVM": make_clustering_model_pipeline(
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                random_state=42,
            ),
            n_clusters=n_clusters,
            sampler=sampler,
            use_feature_selection=use_feature_selection,
            k_best=k_best,
        ),

        "Extra Trees": make_clustering_model_pipeline(
            ExtraTreesClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=1,
            ),
            n_clusters=n_clusters,
            sampler=sampler,
            use_feature_selection=use_feature_selection,
            k_best=k_best,
        ),

        "XGBoost": make_clustering_model_pipeline(
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=2,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=1,
                verbosity=0,
            ),
            n_clusters=n_clusters,
            sampler=sampler,
            use_feature_selection=use_feature_selection,
            k_best=k_best,
        ),
    }

    return models

In [15]:
clustering_experiments = {
    "kmeans_k2_no_sampling": {
        "n_clusters": 2,
        "sampler": None,
        "use_feature_selection": False,
        "description": "KMeans k=2, no oversampling, all enhanced features",
    },

    "kmeans_k3_no_sampling": {
        "n_clusters": 3,
        "sampler": None,
        "use_feature_selection": False,
        "description": "KMeans k=3, no oversampling, all enhanced features",
    },

    "kmeans_k4_no_sampling": {
        "n_clusters": 4,
        "sampler": None,
        "use_feature_selection": False,
        "description": "KMeans k=4, no oversampling, all enhanced features",
    },

    "kmeans_k3_random_oversampling": {
        "n_clusters": 3,
        "sampler": RandomOverSampler(random_state=42),
        "use_feature_selection": False,
        "description": "KMeans k=3, random oversampling, all enhanced features",
    },

    "kmeans_k4_random_oversampling": {
        "n_clusters": 4,
        "sampler": RandomOverSampler(random_state=42),
        "use_feature_selection": False,
        "description": "KMeans k=4, random oversampling, all enhanced features",
    },

    "kmeans_k3_top30_random_oversampling": {
        "n_clusters": 3,
        "sampler": RandomOverSampler(random_state=42),
        "use_feature_selection": True,
        "description": "KMeans k=3, random oversampling, top 30 selected features",
    },
}

for experiment_name, config in clustering_experiments.items():
    print(experiment_name, "→", config["description"])

kmeans_k2_no_sampling → KMeans k=2, no oversampling, all enhanced features
kmeans_k3_no_sampling → KMeans k=3, no oversampling, all enhanced features
kmeans_k4_no_sampling → KMeans k=4, no oversampling, all enhanced features
kmeans_k3_random_oversampling → KMeans k=3, random oversampling, all enhanced features
kmeans_k4_random_oversampling → KMeans k=4, random oversampling, all enhanced features
kmeans_k3_top30_random_oversampling → KMeans k=3, random oversampling, top 30 selected features


## 6. LOPO Evaluation of Clustering-Enhanced Models

The clustering-enhanced models are evaluated using Leave-One-Participant-Out cross-validation. In each fold, all samples from one participant are held out as the test set.

The complete pipeline is fitted only on the training participants. This includes imputation, scaling, KMeans clustering, optional feature selection, optional oversampling and classifier training. The held-out participant is only transformed using the fitted pipeline and then predicted.

The final performance metrics are computed from pooled predictions across all LOPO folds.

In [17]:
def evaluate_lopo_clustering_models(
    X,
    y,
    groups,
    experiments,
    target_name,
    k_best=30,
):
    all_results = []
    all_predictions = {}

    for experiment_name, experiment_config in experiments.items():
        print("\n" + "=" * 90)
        print(f"Target: {target_name}")
        print(f"Experiment: {experiment_name}")
        print(experiment_config["description"])
        print("=" * 90)

        models = build_clustering_ml_models(
            n_clusters=experiment_config["n_clusters"],
            sampler=experiment_config["sampler"],
            use_feature_selection=experiment_config["use_feature_selection"],
            k_best=k_best,
        )

        for model_name, model in models.items():
            print(f"Evaluating {model_name}...")

            y_pred = cross_val_predict(
                estimator=model,
                X=X,
                y=y,
                groups=groups,
                cv=logo,
                n_jobs=1,
            )

            prediction_key = (
                target_name,
                experiment_name,
                model_name,
            )

            all_predictions[prediction_key] = y_pred

            cm = confusion_matrix(
                y,
                y_pred,
                labels=binary_label_order_numeric,
            )

            tn, fp, fn, tp = cm.ravel()

            result = {
                "target": target_name,
                "experiment": experiment_name,
                "model": model_name,
                "n_clusters": experiment_config["n_clusters"],
                "sampling": "random_oversampling"
                if experiment_config["sampler"] is not None
                else "no_sampling",
                "feature_selection": experiment_config["use_feature_selection"],
                "accuracy": accuracy_score(y, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y, y_pred),
                "macro_f1": f1_score(
                    y,
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
                "weighted_f1": f1_score(
                    y,
                    y_pred,
                    average="weighted",
                    zero_division=0,
                ),
                "precision_low": precision_score(
                    y,
                    y_pred,
                    pos_label=0,
                    zero_division=0,
                ),
                "recall_low": recall_score(
                    y,
                    y_pred,
                    pos_label=0,
                    zero_division=0,
                ),
                "precision_high": precision_score(
                    y,
                    y_pred,
                    pos_label=1,
                    zero_division=0,
                ),
                "recall_high": recall_score(
                    y,
                    y_pred,
                    pos_label=1,
                    zero_division=0,
                ),
                "true_low": int((y == 0).sum()),
                "true_high": int((y == 1).sum()),
                "predicted_low": int((y_pred == 0).sum()),
                "predicted_high": int((y_pred == 1).sum()),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
                "n_samples": len(y),
                "n_groups": groups.nunique(),
            }

            all_results.append(result)

    results_df = (
        pd.DataFrame(all_results)
        .sort_values(
            by=["target", "macro_f1", "balanced_accuracy"],
            ascending=[True, False, False],
        )
        .reset_index(drop=True)
    )

    return results_df, all_predictions

In [18]:
valence_clustering_results, valence_clustering_predictions = evaluate_lopo_clustering_models(
    X=X_valence_binary,
    y=y_valence_binary,
    groups=groups_valence_binary,
    experiments=clustering_experiments,
    target_name="valence_binary",
    k_best=30,
)

display(valence_clustering_results)


Target: valence_binary
Experiment: kmeans_k2_no_sampling
KMeans k=2, no oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...

Target: valence_binary
Experiment: kmeans_k3_no_sampling
KMeans k=3, no oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...

Target: valence_binary
Experiment: kmeans_k4_no_sampling
KMeans k=4, no oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...

Target: valence_binary
Experiment: kmeans_k3_random_oversampling
KMeans k=3, random oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...


,target,experiment,model,n_clusters,sampling,feature_selection,accuracy,balanced_accuracy,macro_f1,weighted_f1,precision_low,recall_low,precision_high,recall_high,true_low,true_high,predicted_low,predicted_high,tn,fp,fn,tp,n_samples,n_groups
0,valence_binary,kmeans_k3_no_sampling,RBF SVM,3,no_sampling,False,0.718543,0.687659,0.676386,0.725118,0.514286,0.613636,0.827411,0.761682,88,214,105,197,54,34,51,163,302,96
1,valence_binary,kmeans_k4_no_sampling,RBF SVM,4,no_sampling,False,0.715232,0.688668,0.675220,0.722781,0.509259,0.625000,0.829897,0.752336,88,214,108,194,55,33,53,161,302,96
2,valence_binary,kmeans_k2_no_sampling,RBF SVM,2,no_sampling,False,0.711921,0.679641,0.668772,0.718650,0.504762,0.602273,0.822335,0.757009,88,214,105,197,53,35,52,162,302,96
3,valence_binary,kmeans_k3_top30_random_oversampling,XGBoost,3,random_oversampling,True,0.708609,0.670614,0.662141,0.714418,0.500000,0.579545,0.815000,0.761682,88,214,102,200,51,37,51,163,302,96
4,valence_binary,kmeans_k3_top30_random_oversampling,Logistic Regression,3,random_oversampling,True,0.692053,0.679004,0.658234,0.703089,0.478992,0.647727,0.830601,0.710280,88,214,119,183,57,31,62,152,302,96
5,valence_binary,kmeans_k4_random_oversampling,XGBoost,4,random_oversampling,False,0.711921,0.652878,0.652364,0.712397,0.505618,0.511364,0.798122,0.794393,88,214,89,213,45,43,44,170,302,96
6,valence_binary,kmeans_k2_no_sampling,Extra Trees,2,no_sampling,False,0.725166,0.645497,0.651294,0.718257,0.533333,0.454545,0.788546,0.836449,88,214,75,227,40,48,35,179,302,96
7,valence_binary,kmeans_k3_random_oversampling,RBF SVM,3,random_oversampling,False,0.698675,0.656914,0.649621,0.704319,0.485149,0.556818,0.805970,0.757009,88,214,101,201,49,39,52,162,302,96
8,valence_binary,kmeans_k3_top30_random_oversampling,RBF SVM,3,random_oversampling,True,0.685430,0.664295,0.647556,0.695760,0.469565,0.613636,0.818182,0.714953,88,214,115,187,54,34,61,153,302,96
9,valence_binary,kmeans_k4_random_oversampling,RBF SVM,4,random_oversampling,False,0.695364,0.654577,0.646784,0.701437,0.480392,0.556818,0.805000,0.752336,88,214,102,200,49,39,53,161,302,96


In [19]:
display(
    valence_clustering_results
    .sort_values(
        by=["macro_f1", "balanced_accuracy"],
        ascending=False,
    )
    .head(15)
)

,target,experiment,model,n_clusters,sampling,feature_selection,accuracy,balanced_accuracy,macro_f1,weighted_f1,precision_low,recall_low,precision_high,recall_high,true_low,true_high,predicted_low,predicted_high,tn,fp,fn,tp,n_samples,n_groups
0,valence_binary,kmeans_k3_no_sampling,RBF SVM,3,no_sampling,False,0.718543,0.687659,0.676386,0.725118,0.514286,0.613636,0.827411,0.761682,88,214,105,197,54,34,51,163,302,96
1,valence_binary,kmeans_k4_no_sampling,RBF SVM,4,no_sampling,False,0.715232,0.688668,0.675220,0.722781,0.509259,0.625000,0.829897,0.752336,88,214,108,194,55,33,53,161,302,96
2,valence_binary,kmeans_k2_no_sampling,RBF SVM,2,no_sampling,False,0.711921,0.679641,0.668772,0.718650,0.504762,0.602273,0.822335,0.757009,88,214,105,197,53,35,52,162,302,96
3,valence_binary,kmeans_k3_top30_random_oversampling,XGBoost,3,random_oversampling,True,0.708609,0.670614,0.662141,0.714418,0.500000,0.579545,0.815000,0.761682,88,214,102,200,51,37,51,163,302,96
4,valence_binary,kmeans_k3_top30_random_oversampling,Logistic Regression,3,random_oversampling,True,0.692053,0.679004,0.658234,0.703089,0.478992,0.647727,0.830601,0.710280,88,214,119,183,57,31,62,152,302,96
5,valence_binary,kmeans_k4_random_oversampling,XGBoost,4,random_oversampling,False,0.711921,0.652878,0.652364,0.712397,0.505618,0.511364,0.798122,0.794393,88,214,89,213,45,43,44,170,302,96
6,valence_binary,kmeans_k2_no_sampling,Extra Trees,2,no_sampling,False,0.725166,0.645497,0.651294,0.718257,0.533333,0.454545,0.788546,0.836449,88,214,75,227,40,48,35,179,302,96
7,valence_binary,kmeans_k3_random_oversampling,RBF SVM,3,random_oversampling,False,0.698675,0.656914,0.649621,0.704319,0.485149,0.556818,0.805970,0.757009,88,214,101,201,49,39,52,162,302,96
8,valence_binary,kmeans_k3_top30_random_oversampling,RBF SVM,3,random_oversampling,True,0.685430,0.664295,0.647556,0.695760,0.469565,0.613636,0.818182,0.714953,88,214,115,187,54,34,61,153,302,96
9,valence_binary,kmeans_k4_random_oversampling,RBF SVM,4,random_oversampling,False,0.695364,0.654577,0.646784,0.701437,0.480392,0.556818,0.805000,0.752336,88,214,102,200,49,39,53,161,302,96


In [20]:
arousal_clustering_results, arousal_clustering_predictions = evaluate_lopo_clustering_models(
    X=X_arousal_binary,
    y=y_arousal_binary,
    groups=groups_arousal_binary,
    experiments=clustering_experiments,
    target_name="arousal_binary",
    k_best=30,
)

display(arousal_clustering_results)


Target: arousal_binary
Experiment: kmeans_k2_no_sampling
KMeans k=2, no oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...

Target: arousal_binary
Experiment: kmeans_k3_no_sampling
KMeans k=3, no oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...

Target: arousal_binary
Experiment: kmeans_k4_no_sampling
KMeans k=4, no oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...

Target: arousal_binary
Experiment: kmeans_k3_random_oversampling
KMeans k=3, random oversampling, all enhanced features
Evaluating Dummy most frequent...
Evaluating Logistic Regression...
Evaluating RBF SVM...
Evaluating Extra Trees...
Evaluating XGBoost...


,target,experiment,model,n_clusters,sampling,feature_selection,accuracy,balanced_accuracy,macro_f1,weighted_f1,precision_low,recall_low,precision_high,recall_high,true_low,true_high,predicted_low,predicted_high,tn,fp,fn,tp,n_samples,n_groups
0,arousal_binary,kmeans_k4_random_oversampling,Extra Trees,4,random_oversampling,False,0.549407,0.548898,0.548836,0.549534,0.528455,0.537190,0.569231,0.560606,121,132,123,130,65,56,58,74,253,95
1,arousal_binary,kmeans_k3_random_oversampling,RBF SVM,3,random_oversampling,False,0.541502,0.542700,0.541495,0.541416,0.518797,0.570248,0.566667,0.515152,121,132,133,120,69,52,64,68,253,95
2,arousal_binary,kmeans_k4_random_oversampling,RBF SVM,4,random_oversampling,False,0.541502,0.542700,0.541495,0.541416,0.518797,0.570248,0.566667,0.515152,121,132,133,120,69,52,64,68,253,95
3,arousal_binary,kmeans_k4_no_sampling,Extra Trees,4,no_sampling,False,0.541502,0.541667,0.541323,0.541717,0.519685,0.545455,0.563492,0.537879,121,132,127,126,66,55,61,71,253,95
4,arousal_binary,kmeans_k3_random_oversampling,XGBoost,3,random_oversampling,False,0.541502,0.541667,0.541323,0.541717,0.519685,0.545455,0.563492,0.537879,121,132,127,126,66,55,61,71,253,95
5,arousal_binary,kmeans_k2_no_sampling,Extra Trees,2,no_sampling,False,0.541502,0.540978,0.540921,0.541631,0.520325,0.528926,0.561538,0.553030,121,132,123,130,64,57,59,73,253,95
6,arousal_binary,kmeans_k4_no_sampling,RBF SVM,4,no_sampling,False,0.537549,0.537879,0.537434,0.537752,0.515625,0.545455,0.560000,0.530303,121,132,128,125,66,55,62,70,253,95
7,arousal_binary,kmeans_k2_no_sampling,RBF SVM,2,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95
8,arousal_binary,kmeans_k3_no_sampling,RBF SVM,3,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95
9,arousal_binary,kmeans_k3_no_sampling,Extra Trees,3,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95


In [21]:
display(
    arousal_clustering_results
    .sort_values(
        by=["macro_f1", "balanced_accuracy"],
        ascending=False,
    )
    .head(15)
)

,target,experiment,model,n_clusters,sampling,feature_selection,accuracy,balanced_accuracy,macro_f1,weighted_f1,precision_low,recall_low,precision_high,recall_high,true_low,true_high,predicted_low,predicted_high,tn,fp,fn,tp,n_samples,n_groups
0,arousal_binary,kmeans_k4_random_oversampling,Extra Trees,4,random_oversampling,False,0.549407,0.548898,0.548836,0.549534,0.528455,0.537190,0.569231,0.560606,121,132,123,130,65,56,58,74,253,95
1,arousal_binary,kmeans_k3_random_oversampling,RBF SVM,3,random_oversampling,False,0.541502,0.542700,0.541495,0.541416,0.518797,0.570248,0.566667,0.515152,121,132,133,120,69,52,64,68,253,95
2,arousal_binary,kmeans_k4_random_oversampling,RBF SVM,4,random_oversampling,False,0.541502,0.542700,0.541495,0.541416,0.518797,0.570248,0.566667,0.515152,121,132,133,120,69,52,64,68,253,95
3,arousal_binary,kmeans_k4_no_sampling,Extra Trees,4,no_sampling,False,0.541502,0.541667,0.541323,0.541717,0.519685,0.545455,0.563492,0.537879,121,132,127,126,66,55,61,71,253,95
4,arousal_binary,kmeans_k3_random_oversampling,XGBoost,3,random_oversampling,False,0.541502,0.541667,0.541323,0.541717,0.519685,0.545455,0.563492,0.537879,121,132,127,126,66,55,61,71,253,95
5,arousal_binary,kmeans_k2_no_sampling,Extra Trees,2,no_sampling,False,0.541502,0.540978,0.540921,0.541631,0.520325,0.528926,0.561538,0.553030,121,132,123,130,64,57,59,73,253,95
6,arousal_binary,kmeans_k4_no_sampling,RBF SVM,4,no_sampling,False,0.537549,0.537879,0.537434,0.537752,0.515625,0.545455,0.560000,0.530303,121,132,128,125,66,55,62,70,253,95
7,arousal_binary,kmeans_k2_no_sampling,RBF SVM,2,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95
8,arousal_binary,kmeans_k3_no_sampling,RBF SVM,3,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95
9,arousal_binary,kmeans_k3_no_sampling,Extra Trees,3,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95


In [22]:
clustering_results = pd.concat(
    [
        valence_clustering_results,
        arousal_clustering_results,
    ],
    ignore_index=True,
)

clustering_results = (
    clustering_results
    .sort_values(
        by=["target", "macro_f1", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

display(clustering_results)

,target,experiment,model,n_clusters,sampling,feature_selection,accuracy,balanced_accuracy,macro_f1,weighted_f1,precision_low,recall_low,precision_high,recall_high,true_low,true_high,predicted_low,predicted_high,tn,fp,fn,tp,n_samples,n_groups
0,arousal_binary,kmeans_k4_random_oversampling,Extra Trees,4,random_oversampling,False,0.549407,0.548898,0.548836,0.549534,0.528455,0.537190,0.569231,0.560606,121,132,123,130,65,56,58,74,253,95
1,arousal_binary,kmeans_k3_random_oversampling,RBF SVM,3,random_oversampling,False,0.541502,0.542700,0.541495,0.541416,0.518797,0.570248,0.566667,0.515152,121,132,133,120,69,52,64,68,253,95
2,arousal_binary,kmeans_k4_random_oversampling,RBF SVM,4,random_oversampling,False,0.541502,0.542700,0.541495,0.541416,0.518797,0.570248,0.566667,0.515152,121,132,133,120,69,52,64,68,253,95
3,arousal_binary,kmeans_k4_no_sampling,Extra Trees,4,no_sampling,False,0.541502,0.541667,0.541323,0.541717,0.519685,0.545455,0.563492,0.537879,121,132,127,126,66,55,61,71,253,95
4,arousal_binary,kmeans_k3_random_oversampling,XGBoost,3,random_oversampling,False,0.541502,0.541667,0.541323,0.541717,0.519685,0.545455,0.563492,0.537879,121,132,127,126,66,55,61,71,253,95
5,arousal_binary,kmeans_k2_no_sampling,Extra Trees,2,no_sampling,False,0.541502,0.540978,0.540921,0.541631,0.520325,0.528926,0.561538,0.553030,121,132,123,130,64,57,59,73,253,95
6,arousal_binary,kmeans_k4_no_sampling,RBF SVM,4,no_sampling,False,0.537549,0.537879,0.537434,0.537752,0.515625,0.545455,0.560000,0.530303,121,132,128,125,66,55,62,70,253,95
7,arousal_binary,kmeans_k2_no_sampling,RBF SVM,2,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95
8,arousal_binary,kmeans_k3_no_sampling,RBF SVM,3,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95
9,arousal_binary,kmeans_k3_no_sampling,Extra Trees,3,no_sampling,False,0.529644,0.529959,0.529527,0.529850,0.507812,0.537190,0.552000,0.522727,121,132,128,125,65,56,63,69,253,95


In [ ]:
best_clustering_models_per_target = (
    clustering_results
    .sort_values(
        by=["target", "macro_f1", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .groupby("target")
    .head(10)
    .reset_index(drop=True)
)

display(best_clustering_models_per_target)

In [ ]:
best_clustering_model_per_target = (
    clustering_results
    .sort_values(
        by=["target", "macro_f1", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .groupby("target")
    .head(1)
    .reset_index(drop=True)
)

display(best_clustering_model_per_target)

In [ ]:
for target_name in clustering_results["target"].unique():
    plot_df = (
        clustering_results[
            clustering_results["target"] == target_name
        ]
        .sort_values("macro_f1", ascending=False)
        .head(15)
        .sort_values("macro_f1", ascending=True)
    )

    labels = (
        plot_df["model"]
        + "\n"
        + plot_df["experiment"]
    )

    plt.figure(figsize=(10, 7))
    plt.barh(labels, plot_df["macro_f1"])

    plt.xlabel("Macro F1")
    plt.ylabel("Model / Experiment")
    plt.title(f"Top Clustering-Enhanced LOPO Models: {target_name}")
    plt.grid(axis="x", alpha=0.3)

    for i, value in enumerate(plot_df["macro_f1"]):
        plt.text(
            value + 0.005,
            i,
            f"{value:.3f}",
            va="center",
            fontsize=9,
        )

    plt.xlim(0, max(0.55, plot_df["macro_f1"].max() + 0.08))
    plt.tight_layout()
    plt.show()

In [ ]:
CLUSTERING_RESULTS_PATH = OUTPUT_DIR / "lopo_binary_clustering_ml_results.csv"

clustering_results.to_csv(CLUSTERING_RESULTS_PATH, index=False)

print("Saved clustering-enhanced model results to:")
print(CLUSTERING_RESULTS_PATH)

The clustering-enhanced models did not provide consistent improvement over the previous global binary models. Although valence performance was reasonable, the best models were still driven by the supervised classifier rather than by a clear benefit from the clustering representation. Arousal performance remained weak, suggesting that unsupervised KMeans clusters do not capture response patterns that are sufficiently aligned with self-reported arousal.

Therefore, the next modeling step focuses on a more structured two-layer architecture based on modality-level model stacking.